# Nodes and Terminals

All current-carrying electrical equipment inheriting from the `ConductingEquipment` class are defined as being connected to a Terminal object associated with a `ConnectivityNode`. Generators, shunt capacitors, shunt reactors, and loads are connected to one Terminal object associated with a single `ConnectivityNode`. Lines, cables, series capacitors, and series reactors have two Terminal objects associated with either end of the branch. Rather than defining from/to buses, CIM identifies the end of a branch by setting the `ACDCTerminal.sequenceNumber` attribute of a terminal to values of 1 or 2 to specify the particular end. Transformers are defined using two or three terminals, with the `ACDCTerminal`. `sequenceNumber` attribute used to designate whether the Terminal is associated with the primary, secondary, or tertiary windings. Split-phase distribution transformers also use three terminals, but the two low-side terminals are associated with a single `ConnectivityNode`, as shown in Figure below. 

![Alt text](images/Fig_8_Chapter_5.png)

Although this approach may seem excessively detailed and complicated compared to bus-branch modeling used by many analysis tools, the set of `ConnectivityNode` and Terminal objects provide a node-edge graph structure built into the power system network model. This graph structure can then be used for highly efficient topology processing and mapping of the electrical network using the `TopologicalNode` and `TopologicalIsland` objects associated with each `ConnectivityNode`. 

Consider a data mapping problem in which DERs need to be associated with substation breakers: The entire analysis could be accomplished with only a topological model specifying the association of `ConnectivityNode` and Terminal objects to their associated DERs, lines, transformers, and switches. The node-edge graph structure built into the CIM model can then be used to determine which DERs are connected to which breakers by building a spanning tree to determine the associated Feeder or `TopologicalIsland` in which the DER is contained. Detailed modeling of other aspects of the power system model (such as would be needed to solve a full power flow solution) is not required, and consequently, a very simple CIM profile could be adopted with far less complexity than would be needed for full model exchange between traditional analysis software packages. The UML class diagram showing the detailed associations between nodes, terminals, power system equipment, and how they can be organized into a particular feeder and substation is showin in Figure below.

In [12]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [13]:
diagram_text = utils.get_mermaid([cim.Terminal, cim.ConductingEquipment, cim.ConnectivityNode, cim.ACDCTerminal, cim.Feeder, cim.Substation, cim.Equipment])
Mermaid(diagram_text)

Associated with each Terminal object are measurements of power system parameters. A `Measurement` object is used to represent any direct measurement, calculated value, or non-measured non-calculated value. Examples of possible `Measurement` objects include the position of a transformer tap, current measured by a current transformer (CT), calculated MW flow of a line, the oil temperature of a transformer, or whether a substation door is open. The type of a measurement is specified as a text string through the `Measurement.measurementType` attribute. CIM does not directly specify how measurements are named or defined. Rather, it provides high-level classes for the overall type of `Measurement`, such as `Analog` (e.g. voltage), `Discrete` (e.g. breaker status), and `Accumulator` (e.g. metered kWh). 

In [27]:
diagram_text = utils.get_mermaid([cim.Terminal, cim.Measurement, cim.PowerSystemResource, cim.ConductingEquipment, cim.ConnectivityNode, cim.ACDCTerminal,  cim.Equipment, cim.Switch, cim.TopologicalNode, cim.TopologicalIsland])
Mermaid(diagram_text)

`Measurement` objects associated with power flow quantities (e.g. MW flow at the end of a line) are associated with the `ACDCTerminal` at which the measurement is taken. `Measurement` objects relating to properties of the equipment itself (e.g. oil temperature) are generally associated with the `PowerSystemResource` object for that piece of equipment. Figure above illustrates how a Measurement is associated to the `MeasurementValue`, `Terminal`, and `PowerSystemResource`.  Because CIM objects inherit all of associations of their parent classes, the Measurement can be associated with a specific device, such as a `Switch`.

Figure 11 below highlights the different kinds of measurements available within CIM, including the classes:
* Analog – used for voltage, power, current and other continuous measurements
* Discrete – used for switch open/closed positions, transformer tap positions, etc.
* Accumulator – used for time-integrated measurements (such as meter kWh)


In [39]:
diagram_text = utils.get_mermaid([cim.Terminal, cim.Measurement, cim.PowerSystemResource, cim.Measurement, cim.Analog, cim.AnalogValue, cim.AnalogLimit, cim.AnalogLimitSet, cim.Accumulator, cim.AccumulatorLimit, cim.AccumulatorLimitSet, cim.AccumulatorValue, cim.StringMeasurement, cim.StringMeasurementValue, cim.MeasurementValue,cim.Discrete, cim.DiscreteValue, cim.ValueAliasSet, cim.ValueToAlias, cim.MeasurementValueSource, cim.MeasurementValueQuality, cim.Quality61850, cim.IOPoint])
Mermaid(diagram_text)

Some examples are explained next.


In [40]:
from cimgraph.databases import ConnectionParameters, XMLFile
from cimgraph.models import FeederModel

from cimgraph import utils
from mermaid import Mermaid

import cimgraph.data_profile.cimhub_2023 as cim

In [41]:
params = ConnectionParameters(filename="../sample_models/ieee13.xml",
            cim_profile='cimhub_2023', iec61970_301=8)
xml_file = XMLFile(params)
network = FeederModel(connection=xml_file, container=cim.Feeder())

Example 1: Which topological island is node with mrid 0124E881-B82D-4206-BBDF-37D585159872 part of?

In [43]:
results = []

# Retrieve the node with the specified UUID from the graph
node = network.get_object(mRID = '0124E881-B82D-4206-BBDF-37D585159872')

# Retrieve the TopologicalNode associated with the ConnectivityNode
topological_node = node.TopologicalNode

# Retrieve the TopologicalIsland associated with the TopologicalNode
topological_island = topological_node.TopologicalIsland

# Append the name of the TopologicalIsland to the results list
results.append(topological_island.name)

# Print the results which contains the name of the topological island
print(results)

['ieee13nodeckt_Island']


Example 2: What is the length of the line from node bus 632 to node bus 645?

In [44]:
from_name = '632'
to_name = '645'
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '632'
    if from_name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment is an instance of ACLineSegment
            if isinstance(equipment, cim.ACLineSegment):
                # Loop through all Terminals of the ConductingEquipment
                for far_terminal in equipment.Terminals:
                    # Get the far-end ConnectivityNode associated with the terminal
                    far_node = far_terminal.ConnectivityNode
                    # Check if the far-end node's name contains '645'
                    if to_name in far_node.name:
                        # Append the length of the ACLineSegment to the results list
                        results.append(equipment.length)
        # Break after the first matching node to optimize performance
        break

# Remove duplicates by converting results list to a set
results = set(results)

# Print the results which contains the lengths of the lines from node bus '632' to node bus '645'
print(results)

{152.4}


Example 3: What types of equipment are connected to node bus 634?

In [45]:
name = '634'
results = []
        


# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string
    if name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Append the type (class name) of the ConductingEquipment to the results list
            results.append(equipment.__class__.__name__)
        # Break after the first matching node to optimize performance
        break

# Output the results which contains the types of equipment connected to node bus '634'
print(results)

['PowerElectronicsConnection', 'PowerElectronicsConnection', 'PowerElectronicsConnection', 'PowerTransformer', 'EnergyConsumer', 'EnergyConsumer', 'EnergyConsumer']


Example 4: What is the real power of load connected to bus 634?

In [46]:
name = '634'
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '634'
    if name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment is an instance of EnergyConsumer
            if isinstance(equipment, cim.EnergyConsumer):
                # Append the real power (p) to the results list
                results.append(equipment.p)
        # Break after the first matching node to optimize performance
        break

# Print the results which contains the real power of loads connected to node bus '634'
print(results)

[160000.0, 120000.0, 120000.0]


Example 5: What are the other nodes connected to bus 634?

In [47]:
name = '634'
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '634'
    if name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Loop through all Terminals of the ConductingEquipment
            for far_terminal in equipment.Terminals:
                # Get the far-end ConnectivityNode associated with the terminal
                far_node = far_terminal.ConnectivityNode
                # Append the far-end node's name to the results list
                results.append(far_node.name)
        # Break after the first matching node to optimize performance
        break

# Remove duplicates by converting results list to a set
results = set(results)

# Print the results which contains the names of other nodes connected to node bus '634'
print(results)

{'634', 'xf1'}


Example 6: What type of equipment connects nodes 670 and house?

In [48]:
from_name = '670'
to_name = 'house'
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the from_name
    if from_name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Loop through all Terminals of the ConductingEquipment
            for far_terminal in equipment.Terminals:
                # Get the far-end ConnectivityNode associated with the terminal
                far_node = far_terminal.ConnectivityNode
                # Check if the far-end node's name contains the to_name
                if to_name in far_node.name:
                    # Append the class name of the ConductingEquipment to the results list
                    results.append(equipment.__class__.__name__)
        # Break after the first matching node to optimize performance
        break

# Remove duplicates by converting results list to a set
results = set(results)

# Print the results which contains the types of equipment connecting two different nodes 
print(results)

{'PowerTransformer'}


Example 7: What is the uuid of node bus 634?

In [49]:
results = []
name = '634'

# The final attribute is cim.ConnectivityNode.mRID

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '634'
    if name in node.name:
        # Append the mRID (UUID) of the node to the results list
        results.append(node.mRID)
        # Break after the first matching ConnectivityNode to optimize performance
        break

# Print the results which contains the UUID of the node bus '634'
print(results)

['0DCC57AF-F4FA-457D-BB24-2EFDA9865A1A']


Example 8: What are the voltage limits set for node 632? What is the maximum allowable voltage for node 632?

In [50]:
results = []
max_result = []
name = '632'

# The final attribute is cim.VoltageLimit.value
# graph path traversal is ConnectivityNode -> List[OperationalLimitSet] -> List[OperationalLimitValue]

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '632'
    if name in node.name:
        # Loop through OperationalLimitSets associated with the ConnectivityNode
        for limit in node.OperationalLimitSet:
            # Loop through OperationalLimitValues associated with the OperationalLimitSet
            for limit_value in limit.OperationalLimitValue:
                # Check if the OperationalLimitValue is an instance of VoltageLimit
                if isinstance(limit_value, cim.VoltageLimit):
                    # Append the limit name and value to the results list
                    results.append([limit_value.name, limit_value.value])
                    # Append the limit value (converted to float) to the max_result list
                    max_result.append(float(limit_value.value))
        # Break after the first matching ConnectivityNode to optimize performance
        break

# Print the collected voltage limits associated with node bus '632'
print(results)

# Print the maximum allowable voltage for the node bus '632'
print(max(max_result))

[['OpLimV_4.1600_RangeAHi', 4368.0], ['OpLimV_4.1600_RangeALo', 3952.0], ['OpLimV_4.1600_RangeBHi', 4402.6665], ['OpLimV_4.1600_RangeBLo', 3813.3335]]
4402.6665


Example 9: What phases are associated with bus node 634?

In [51]:
results = []
name = "house"

# The final attribute is objects with typing of SinglePhaseKind or OrderPhaseCodeKind
# graph path traversal is ConnectivityNode -> List[Terminal] -> ConductingEquipment -> phase

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '633'
    if name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            
            # # Check if the ConductingEquipment is an instance of ACLineSegment
            # if isinstance(equipment, cim.ACLineSegment):
            #     # Loop through ACLineSegmentPhases associated with ACLineSegment
            #     for ac_line_segment_phase in equipment.ACLineSegmentPhases:
            #         # Append the phase to the results list
            #         results.append(ac_line_segment_phase.phase)
            
            # Check if the ConductingEquipment is an instance of PowerTransformer
            if isinstance(equipment, cim.PowerTransformer):
                # Loop through TransformerTanks associated with PowerTransformer
                for transformer_tank in equipment.TransformerTanks:
                    # Loop through TransformerTankEnds associated with TransformerTank
                    for tank_end in transformer_tank.TransformerTankEnds:
                        # Check if the TransformerTankEnd's Terminal matches the current Terminal
                        if tank_end.Terminal == terminal:
                            # Append the ordered phases to the results list
                            results.append(str(tank_end.orderedPhases))
            
            # # Check if the ConductingEquipment is an instance of EnergyConsumer
            # elif isinstance(equipment, cim.EnergyConsumer):
            #     # Loop through EnergyConsumerPhases associated with EnergyConsumer
            #     for energy_consumer_phase in equipment.EnergyConsumerPhase:
            #         # Append the phase to the results list
            #         results.append(energy_consumer_phase.phase)

# If no phases were found, it is likely a three-phase node
if not results:
    results = ['no phases found, it is likely a three-phase node']

# Remove duplicates by converting results list to a set
results = set(results)

# Print the results which contains the phases associated with node bus '634'
print(results)

{'OrderedPhaseCodeKind.Ns2', 'OrderedPhaseCodeKind.s1N'}


Example 10: What are the xy location coordinates for node bus 634?

In [53]:
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string
    if name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment has a Location attribute
            if equipment.Location is not None:
                location = equipment.Location
                # Loop through PositionPoints associated with the Location
                for position_point in location.PositionPoints:
                    # Check if the sequenceNumber matches the terminal's sequenceNumber
                    if position_point.sequenceNumber == terminal.sequenceNumber:
                        # Append the x and y coordinates to the results list
                        coordinates = dict()
                        coordinates['x'] = position_point.xPosition
                        coordinates['y'] = position_point.yPosition
                        results.append(coordinates)
            # Break if results were found to prevent unnecessary computations
            if results:
                break

# Print the results which contains the xy coordinates for node bus '634'
print(results)

[{'x': '200', 'y': '190'}]


Example 11: What is the name of the inverters connected to node bus 634?

In [54]:
results = []
name = '634'
# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string
    if name == node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment is an instance of PowerElectronicsConnection (e.g., Inverters)
            if isinstance(equipment, cim.PowerElectronicsConnection):
                # Append the name of the PowerElectronicsConnection to the results list
                results.append(equipment.name)
        # Break after the first matching ConnectivityNode to optimize performance (assuming nodes have unique names)
        break

# Print the results which contains names of the inverters connected to node bus '634'
print(results)

['school', 'school', 'batidle']


Example 12: What is the nominal voltage for node bus 634?


In [55]:
results = []
name = "634"

# The final attribute is cim.BaseVoltage.nominalVoltage
# graph path traversal is ConnectivityNode -> List[Terminal] -> ConductingEquipment -> BaseVoltage

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string
    if name == node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment has a BaseVoltage attribute
            if equipment.BaseVoltage is not None:
                # Append the nominal voltage from BaseVoltage to the results list
                results.append(equipment.BaseVoltage.nominalVoltage)

results = set(results)

# Print the results which contains the nominal voltages for the node bus '634'
print(results)

{480.0}


In [16]:
results = []

# Retrieve the node with the specified UUID from the graph
node = network.get_object(mRID = '0124E881-B82D-4206-BBDF-37D585159872')

# Traverse through all Terminals associated with the ConnectivityNode
for terminal in node.Terminals:
    # Loop through all Measurements associated with the Terminal
    for measurement in terminal.Measurements:
        # Append the mRID of the measurement to the results list
        results.append(measurement.mRID)

# results contains the mRID of all measurements associated with the node
print(results)

['52e4c07c-0611-49cc-aef4-828ead9ab1db', '58eaa5e7-2925-422c-82af-10af887dbb2c', 'd70108e9-7c81-4aa8-9c1d-4a31b5b0cd2b', 'b5d54a83-8b4f-4675-9696-1a6d8ced1d1d', 'f262d228-9aad-4a9f-92ae-4c2724cc79a9', 'aeeecdbd-f3a3-4fbc-8121-5d49a9ee1618']


In [17]:
diagram_text = utils.get_mermaid(cim.ConnectivityNode)
Mermaid(diagram_text)

In [18]:
diagram_text = utils.get_mermaid([cim.Terminal, cim.ConductingEquipment, cim.ConnectivityNode, cim.ACDCTerminal])
Mermaid(diagram_text)

In [19]:
from_name = '632'
to_name = '645'
results = []

# Traverse through all ConnectivityNode instances in the network graph
for node in network.graph[cim.ConnectivityNode].values():
    # Check if the node's name contains the specified string '632'
    if from_name in node.name:
        # Loop through all Terminals associated with the ConnectivityNode
        for terminal in node.Terminals:
            # Get the ConductingEquipment associated with the Terminal
            equipment = terminal.ConductingEquipment
            # Check if the ConductingEquipment is an instance of ACLineSegment
            if isinstance(equipment, cim.ACLineSegment):
                # Loop through all Terminals of the ConductingEquipment
                for far_terminal in equipment.Terminals:
                    # Get the far-end ConnectivityNode associated with the terminal
                    far_node = far_terminal.ConnectivityNode
                    # Check if the far-end node's name contains '645'
                    if to_name in far_node.name:
                        # Append the length of the ACLineSegment to the results list
                        line = equipment
                        results.append(equipment.length)
        # Break after the first matching node to optimize performance
        break
# Remove duplicates by converting results list to a set
results = set(results)

# Print the results which contains the lengths of the lines from node bus '632' to node bus '645'
print(results)

{152.4}


In [20]:
diagram_text = utils.get_mermaid_path(node, 'Terminals[2].ConductingEquipment.Terminals[1].ConnectivityNode')
diagram_text = utils.add_mermaid_path(line,'length', diagram_text)
Mermaid(diagram_text)

In [21]:
diagram_text = utils.get_mermaid([cim.ConnectivityNode,cim.TopologicalNode, cim.TopologicalIsland])
Mermaid(diagram_text)

In [22]:
results = []

# Retrieve the node with the specified UUID from the graph
node = network.get_object(mRID = '0124E881-B82D-4206-BBDF-37D585159872')

# Retrieve the TopologicalNode associated with the ConnectivityNode
topological_node = node.TopologicalNode

# Retrieve the TopologicalIsland associated with the TopologicalNode
topological_island = topological_node.TopologicalIsland

# Append the name of the TopologicalIsland to the results list
results.append(topological_island.name)

# Print the results which contains the name of the topological island
print(results)

['ieee13nodeckt_Island']


In [23]:
diagram_text = utils.get_mermaid_path(node,'TopologicalNode.TopologicalIsland')
Mermaid(diagram_text)